**PROJECT INFO**
---------------

| **INFO** | **ENTRY** |
|----------|-----------|
| TITLE | Power BI QA Automation |
| AUTHOR(S) | Aidan Cooney |
| CREATION DATE | 09/23/2025 |
| LAST UPDATED | 09/23/2025 |

**PURPOSE & DESCRIPTION**
-------------------------
The purpose of this script is to automate the Power BI testing process. This script uses a PAT from GitHub to connect to the saved Power BI metadata files and parse them for information on columns used, where they are used, as well as any semantic relationships between tables. 

In order to use this code you will need to store a copy of your PAT in a databricks secret. 

In [0]:
import requests
import re
import json
import pandas as pd

In [0]:
%python
class pbi_evaluate:
    def __init__(self, owner: str, token: str, repo: str, branch: str):
        """
        This initializes the class to a specific repository and 
        branch. A new instance will need to be created to evaluate
        any powerbi report outside of the set repository and branch.
        """
        self.token = token
        self.repo = repo
        self.branch = branch
        self.owner = owner

    def extract_columns(self, data_text: str, data_name: str):
        """
        This function extracts the column information from a single
        tmdl file. This will not extract measures created and not 
        capture any aggregations. 

        Input:
            data_text: string representation of the tmdl file
            data_name: the name of the file (which is also the name 
            of the dataset)
        
        Output:
            cols_dict: dictionary containing information on the columns,
            having key (column, table)
        """
        column_pattern = re.compile(
            r'column (\w+)\s+' +
            r'dataType: ([^\n]+)\s+' +
            r'lineageTag: ([^\n]+)\s+' +
            r'summarizeBy: ([^\n]+)\s+' +
            r'sourceColumn: ([^\n]+)(.*?)' +
            r'(?=column|\\Z)',
            re.DOTALL
        )

        cols_dict = {}

        for match in column_pattern.finditer(data_text):
            col_info = {
                'dataType': match.group(2).strip(),
                'lineageTag': match.group(3).strip(),
                'summarizeBy': match.group(4).strip(),
                'sourceColumn': match.group(5).strip(),
                'annotations': [],
                'changedProperty': None
            }

            name = match.group(1).strip()
            extra = match.group(6)

            annotation_matches = re.findall(r'annotation ([^\n]+) = ([^\n]+)', extra)

            for key, value in annotation_matches:
                col_info['annotations'].append({key.strip(): value.strip()})
            
            changed_property_match = re.search(r'changedProperty = ([^\n]+)', extra)
            
            if changed_property_match:
                col_info['changedProperty'] = changed_property_match.group(1).strip()

            cols_dict[(name, data_name)] = col_info

        return cols_dict
    
    def extract_measures(self, data_text: str, data_name: str):
        """
        This function extracts the measures along with their 
        formulas.

        Input:
            data_text: string representation of the tmdl data file
            data_name: string representation for the name of the data
        
        Output:
            measures: a dictionary that contains the keys (measure_name, data_name) and 
            stores the formula
        """
        pattern = re.compile(
            r"measure ['\"]?([\w\s]+)['\"]?\s*=\s*(.*?)\s+(?:formatString:|lineageTag:)",
            re.DOTALL
        )

        measures = {
            (match.group(1).strip(), data_name): {'formula': match.group(2).strip()}
            for match in pattern.finditer(data_text)
        }

        return measures

    def extract_tables(self, data_text: str):
        """
        This function extracts the table information
        including the schema and catalog the table comes from
        within our datalake. 

        Input:
            data_text: string representation of the tmdl data file

        Output:
            table_info: a dictionary containing the information about the 
            table that contains the key (table_name)
        """
        table_info = {}

        source = re.search(r'Source\s*=\s*([^\(]+)', data_text)

        table_name = re.search(r'partition\s+(.*?)\s*=', data_text).group(1).strip("'")

        catalog_name = None
        schema_name = None
        table_or_view_name = None
        alt_data_source = None
        source_name = None
        databricks_flag = 0

        if source:
            source_name = source.group(1)
            if source_name == 'Databricks.Catalogs':
                catalog_match = re.search(r'\[Name="([^"]+)",Kind="Database"\]', data_text)
                catalog_name = catalog_match.group(1)

                schema_match = re.search(r'\[Name="([^"]+)",Kind="Schema"\]', data_text)
                schema_name = schema_match.group(1)

                table_match = re.search(r'\[Name="([^"]+)",Kind="(Table|View)"\]', data_text)
                table_or_view_name = table_match.group(1)

                databricks_flag = 1

            elif source_name == 'CommonDataService.Database':
                alt_source_loc = re.search(r'CommonDataService.Database\(\"([^"]+)\"\)', data_text)
                alt_data_source = alt_source_loc.group(1)

            elif source_name == 'Value.NativeQuery':
                base_loc = re.search(r'FROM\s+(.*?)(#|"|$)', data_text).group(1)
                joined_locs = re.findall(r'JOIN\s+(.*?)(\s|#|"|$)', data_text)
                alt_data_source = f'{base_loc}'
                for join in joined_locs:
                    alt_data_source += f' JOIN {join[0]}'

            elif source_name == 'Table.Combine':
                comb_tables = re.search(r'Table.Combine\({(.*?)}\)', data_text).group(1).replace('"', '').replace('#', '').split(',')
                alt_data_source = ", ".join(x.strip() for x in comb_tables)

            elif source_name == 'PowerPlatform.Dataflows':
                alt_data_source = re.search(r'entity="(.*?)"', data_text).group(1)
        else:
            source = re.search(r'source\s*=\s*(\{.*?\}|[^\n\(\{]+)', data_text, re.DOTALL)

            if source:
                source_name = source.group(1).strip().replace("'", '')
                alt_data_source = 'Power BI Table'

        table_info[table_name] = {
            'catalog': catalog_name,
            'schema': schema_name,
            'table_or_view': table_or_view_name,
            'source': source_name,
            'alternative_data_source': alt_data_source,
        }

        return table_info
    
    def extract_semantic_model(self, semantic_url, headers):
        """
        This function calls the necesary tools to get the full 
        list of columns, measures, and tables that will then be 
        used to fill in the final data report

        Input:
            semantic_url: Url to the folder containing the data tables
            headers: The necessary headers to access the GitHub API

        Ouput:
            columns, measures, tables: The 3 dictionaries containing the 
            information extracted relative to columns, measures, and tables
        """
        semantic_model_response = requests.get(semantic_url, headers = headers)

        if semantic_model_response.status_code != 200:
            raise Exception('API Connection to GitHub Failed, please ensure your PAT is up to date and the report name is correct.')

        columns = {}
        measures = {}
        tables = {}

        files = semantic_model_response.json()

        for file in files:
            file_text = requests.get(file['_links']['self'], headers = headers).text
            file_name = file['name'].split('.')[0]
            columns.update(self.extract_columns(file_text, file_name))
            measures.update(self.extract_measures(file_text, file_name))
            tables.update(self.extract_tables(file_text))

        return columns, measures, tables
    
    def extract_filters(self, data):
        """
        This function extracts the filters (if any) on the 
        report level. If no filters are found then this returns
        an empty list.

        Input:
            data: The json string at the report level

        Output: 
            report_data: Eiher returns a list of report level filters 
            or an empty list
        """
        try:
            filters = []

            for filter in json.loads(data['filters']):
                expression = filter['expression']

                if 'Column' in expression.keys():
                    filters.append(f'{expression['Column']['Expression']['SourceRef']['Entity']}.{expression['Column']['Property']}')

                elif 'Arithmetic' in expression.keys():
                    filters.append(f'{expression['Arithmetic']['Left']['Measure']['Expression']['SourceRef']['Entity']}.{expression['Arithmetic']['Left']['Measure']['Property']}')
            return filters
        
        except KeyError:
            return []  
        
    def extract_visibility(self, data):
        """
        This function creates the visibility flag. The input
        data can be either a page or a visual.

        Input:
            data: This is a json at either the page or visual level

        Ouput:
            binary 1/0 flag if it is hidden or not (1 for hidden, 0 for not hidden)
        """
        try:
            return data['Visibility']
        except:
            return 0

    def extract_visual_title(self, data):
        """
        This function extracts the title of a visual

        Input:
            data: Json for the visual

        Ouput:
            title: The string value associated with the
            title for the visual, or None if no title is 
            found
        """
        try:
            return data['vcObjects']['title'][0]['properties']['text']['expr']['Literal']['Value']
        
        except:
            return None
        
    def get_catalog_schema(self, tables, table):
        """
        This functin returns the catalog and schema 
        of the specified table.

        Input:
            tables: dictionary of all tables in the report
            table: specific table 

        Ouput:
            catalog, schema: The catalog and Schema of the 
            table, returns None, None if table is not from 
            the datalake
        """
        if tables.get(table, None) == None:
            return None, None, None
        
        if tables.get(table, None).get('table_or_view') == None:
            table_or_view = None
        else:
            table_or_view = tables[table]['table_or_view']

        if tables.get(table, None).get('catalog', None) == None:
            catalog = None
        else:
            catalog = tables[table]['catalog']

        if tables.get(table, None).get('schema', None) == None:
            schema = None
        else:
            schema = tables[table]['schema']
        
        if tables.get(table, None).get('source', None) == None:
            source = None
        else:
            source = tables[table]['source']

        if tables.get(table, None).get('alternative_data_source', None) == None:
            alt_data_source = None
        else:
            alt_data_source = tables[table]['alternative_data_source']
        

        return catalog, schema, table_or_view, source, alt_data_source
        
    def extract_column(self, column, table_map, tables, measures):
        """
        This function extracts the data from the specified column.
        This is the meat of what will be returned as the final deliverable.

        Input:
            column: The json for the column itself
            table_map: The dictionary mapping the table to its alias
            tables: The dictionary of all tables in the report
            measures: The dictionary of all measures in the report

        Output:
            column_data: The dictionary containing the data for the column
        """
        col_type = list(column.keys())[0]

        if col_type == 'Aggregation':
            col_name = column[col_type]['Expression']['Column']['Property']
            table = table_map[column[col_type]['Expression']['Column']['Expression']['SourceRef']['Source']]
            formula = column['Name'].split('(')[0]

            catalog, schema, table_or_view, source, alt_data_source = self.get_catalog_schema(tables, table)
        
        elif col_type == 'Measure':
            col_name = column[col_type]['Property']
            table = table_map[column[col_type]['Expression']['SourceRef']['Source']]
            formula = measures[(col_name, table)]['formula']

            catalog, schema, table_or_view, source, alt_data_source = self.get_catalog_schema(tables, table)

        elif col_type == 'Column':
            col_name = column[col_type]['Property']
            table = table_map[column[col_type]['Expression']['SourceRef']['Source']]
            formula = None

            catalog, schema, table_or_view, source, alt_data_source = self.get_catalog_schema(tables, table)

        return {
            'column': col_name,
            'col_type': col_type,
            'formula': formula,
            'table_name': table,
            'catalog': catalog,
            'schema': schema,
            'table_or_view': table_or_view,
            'source': source,
            'alternative_data_source': alt_data_source,
        }


    def extract_report(self, report_url, headers, columns, measures, tables):
        """
        This function loops through the columns in each visual 
        and extracts the data, then compiles it into a single 
        pandas dataframe.

        Input:
            report_url: The url for the report to be parsed
            headers: The headers for the API call
            columns: The dictionary of all columns in the report
            measures: The dictionary of all measures in the report
            tables: The dictionary of all tables in the report
        
        Output:
            data: The pandas dataframe containing the data for the report
        """
        report_response = requests.get(report_url, headers = headers)

        if report_response.status_code != 200:
            raise Exception('API Connection to GitHub Failed, please ensure that the json file you are trying to parse exists')

        data = report_response.json()
            
        report_filters = self.extract_filters(data)

        visuals_data = []

        for page in data['sections']:
            page_name = page['displayName']

            page_filters = self.extract_filters(page)
            page_hidden = self.extract_visibility(page)

            for visual in page['visualContainers']:
                config = json.loads(visual['config'])

                if 'singleVisual' not in config.keys():
                    continue

                visual_filters = self.extract_filters(visual)
                visual_hidden = self.extract_visibility(visual)

                single_visual = config['singleVisual']

                visual_type = single_visual['visualType']

                title = self.extract_visual_title(single_visual)

                try:
                    table_alias_map = {}
                    for entry in single_visual['prototypeQuery'].get('From', []):
                        table_alias_map[entry['Name']] = entry['Entity']

                    for col in single_visual['prototypeQuery']['Select']:
                        row = self.extract_column(col, table_alias_map, tables, measures)

                        try:
                            display_name = single_visual['columnProperties'][col['Name']]['displayName']
                        except KeyError:
                            display_name = row['column']

                        row.update({
                            'display_name': display_name,
                            'visual_title': title,
                            'visual_type': visual_type,
                            'page_name': page_name,
                            'visual_filters': visual_filters,
                            'page_filters': page_filters,
                            'report_filters': report_filters,
                            'page_hidden': page_hidden,
                            'visual_hidden': visual_hidden,
                        })

                        visuals_data.append(row)
                except:
                    pass

        return pd.DataFrame(visuals_data)

    def unique_columns_used(self, data):
        """
        This function creates a summary table listing
        the columns used and their frequencies.

        Input:
            data: pandas dataframe containing the data parsed
            from the report
        
        Output:
            col_freq: pandas dataframe containing the summary table
        """
        return (
            data
            .groupby(['column', 'table_or_view', 'catalog', 'schema', 'source', 'alternative_data_source'], dropna = False)
            .size()
            .reset_index(name = 'visuals_with_column')
            .sort_values(by = 'visuals_with_column', ascending = False)
            .reset_index(drop = True)
        )

    def unique_tables(self, data):
        """
        This function creates a summary table listing
        the tables used and their frequencies.

        Input:
            data: pandas dataframe containing the data parsed
            from the report

        Output:
            table_freq: pandas dataframe containing the summary table
        """
        return (
            data
            .assign(databricks_flag = (data['source'] == 'Databricks.Catalogs').astype(int))
            .groupby(['table_or_view', 'catalog', 'schema', 'source', 'alternative_data_source', 'databricks_flag'], dropna = False)
            .size()
            .reset_index(name = 'visuals_with_table')
            .sort_values(by = 'visuals_with_table', ascending = False)
            .reset_index(drop = True)
        )

    def extract_relationships(self, relationships_url, headers):
        """
        This function extracts relationships in the semantic model if 
        any could be found.

        Input:
            relationships_url: URL to the relationships.tmdl file
            headers: The headers for the API call

        Output:
            semantic_relationships: A pandas dataframe containing the 
            relationships in the data if any were found
        """
        relationships_response = requests.get(relationships_url, headers = headers)

        semantic_relationships = []

        if relationships_response.status_code == 200:
            relationship = relationships_response.text

            blocks = re.split(r'relationship\s', relationship)

            for block in blocks:
                try:
                    from_cardinality = re.search(r'(?<=fromCardinality:\s).*?(?=\n)', block)
                    to_cardinality = re.search(r'(?<=toCardinality:\s).*?(?=\n)', block)
                    from_col = re.search(r'(?<=fromColumn:\s).*?(?=\n)', block)
                    to_col = re.search(r'(?<=toColumn:\s).*?(?=\n)', block)
                    cross_filtering = re.search(r'(?<=crossFilteringBehavior:\s).*?(?=\n)', block)

                    from_table, from_column = from_col.group(0).replace("'", '').split('.')
                    to_table, to_column = to_col.group(0).replace("'", '').split('.')


                    semantic_relationships.append({
                        'from_table': from_table,
                        'to_table': to_table,
                        'from_column': from_column,
                        'to_column': to_column,
                        'from_cardinality': from_cardinality.group(0) if from_cardinality else 'many',
                        'to_cardinality': to_cardinality.group(0) if to_cardinality else 'one',
                        'cross_filtering': cross_filtering.group(0) if cross_filtering else 'from filters to',
                    })
                except:
                    continue

        return pd.DataFrame(semantic_relationships)
            
    
    def __call__(self, dashboard: str):
        """
        This is the function users will call to run the evaluation.
        It takes as input only the dashboard name, and using that 
        and the set data in the init function, loops through the 
        data (tmdl) files as well as the report.json file to 
        return two pandas dataframes.

        Input:
            dashboard: A string that represents the name of the report

        Output:
            dashboard_data, col_freq_data: dashboard_data is the full dataset 
            for the report. col_data contains the unique column names and
            table names along with the count of visuals they appear in.
        """
        base_url = f'https://api.github.com/repos/{self.owner}/{self.repo}/contents/{dashboard}'
        
        headers = {
            'Authorization': f'token {self.token}',
            'Accept': 'application/vnd.github.v3.raw',
        }

        semantic_model_url = base_url + f'.SemanticModel/definition/tables?ref={self.branch}'
        
        columns, measures, tables = self.extract_semantic_model(semantic_model_url, headers)

        report_url = base_url + f'.Report/report.json?ref={self.branch}'

        report_data = self.extract_report(report_url, headers, columns, measures, tables)

        col_freq_data = self.unique_columns_used(report_data[['column', 'table_or_view', 'catalog', 'schema', 'source', 'alternative_data_source']])

        table_freq_data = self.unique_tables(report_data[['table_or_view', 'catalog', 'schema', 'source', 'alternative_data_source']])

        relationshionships_url = base_url + f'.SemanticModel/definition/relationships.tmdl?ref={self.branch}'

        relationships = self.extract_relationships(relationshionships_url, headers)

        # if len(relationships) > 0:
        #     report_data = report_data.merge(relationships, how = 'left', on = ['column', 'table_name'])

        return report_data, col_freq_data, table_freq_data, relationships

In [0]:
%python
# pbi = pbi_evaluate(
#     token = dbutils.secrets.get(scope = 'INSERT_USERNAME', key = 'INSERT_KEYNAME'),
#     repo = 'REPO-NAME',
#     branch = 'main',
#     owner = 'ORG-NAME'
# )

In [0]:
# data
# col_freq
# table_freq
# semantic_model